# 🤖 LangChain + Bigdata MCP Integration

This notebook demonstrates how your AI agents can interact with **Bigdata.com via MCP (Model Context Protocol)**—a standardized way to connect AI applications to data sources and tools with automatic discovery.

## What This Demonstrates

**Bigdata.com MCP Integration:**
- **Automatic Tool Discovery** → MCP exposes all available tools dynamically; when Bigdata.com adds new capabilities (tearsheets, calendars, screeners), your agent gets them automatically
- **Company Lookup** → Resolve tickers to entity IDs via Knowledge Graph
- **Search** → Query news, filings, transcripts with filters
- **Tearsheets** → Company and country financial profiles
- **Events Calendar** → Earnings dates, conference calls

**Internal Data Integration:**
- Connect to your portfolio databases (positions, transactions, P&L)
- Semantic search over internal research documents via vector stores
- Combine MCP-discovered tools with your internal tools seamlessly

**Framework Flexibility:**
> This demo uses **LangChain** with **langchain-mcp-adapters** and **LangSmith** for observability. The MCP protocol is framework-agnostic—**CrewAI**, **AutoGen**, **Google A2A**, and other frameworks can connect to MCP servers using their respective adapters. The key benefit: one integration, automatic access to all current and future Bigdata.com tools.

---

## What is MCP (Model Context Protocol)?

MCP is an open protocol that standardizes how AI applications connect to data sources and tools. Think of it as "USB for AI" - a universal connector.

**Key Benefits:**
- **Automatic Tool Discovery**: MCP servers expose tools dynamically - no manual updates needed when new tools are added
- **Standardized Interface**: One protocol works across all MCP-compatible tools
- **Stateful Connections**: Efficient communication with long-lived sessions

## Architecture

```
┌─────────────────────────────────────────────────────────┐
│                  LangChain Agent                        │
│  (ReAct Pattern - Reasoning + Acting)                   │
└────────────┬──────────────┬──────────────┬──────────────┘
             │              │              │
             ▼              ▼              ▼
      ┌────────────┐ ┌───────────┐ ┌────────────────┐
      │  Local DB  │ │  FAISS    │ │ Bigdata MCP    │
      │  (SQLite)  │ │  Vector   │ │ Server         │
      │            │ │  Store    │ │                │
      └────────────┘ └───────────┘ └────────────────┘
```

---

## 1️⃣ Install Dependencies

Latest versions of LangChain and LangChain MCP Adapters:

In [1]:
%pip install langchain langchain-openai langchain-community faiss-cpu langchain-mcp-adapters python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## 2️⃣ Import Libraries

**LangChain**: ReAct agent and tool-calling

**LangChain MCP Adapters**: Bridge between LangChain and MCP servers

**langgraph_core** (reusable): Environment, `create_financial_database`, `create_vector_store`, local tools (`get_database_tools`, `get_vectorstore_tools`), and display helpers (`display_query`, `display_response`, `display_tools_used`, `display_citations`)

In [2]:
import os
import json
import sqlite3
import random
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from dotenv import load_dotenv

# Display utilities for Jupyter
from IPython.display import display, Markdown, HTML
import html as html_lib

# LangChain
from langchain.tools import tool
from langchain.agents import create_agent as langchain_create_agent
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# MCP integration
from langchain_mcp_adapters.client import MultiServerMCPClient

# Reusable core (langgraph_core): environment, data sources, display
import sys
sys.path.append(".")
from langgraph_core import (
    setup_environment,
    create_financial_database,
    create_vector_store,
    get_database_tools,
    get_vectorstore_tools,
    display_query,
    display_response,
    display_tools_used,
    display_citations,
)
load_dotenv()
print("✅ Libraries imported successfully")

/Users/bakulkumarkakadiya/dev/github/bigdata-cookbook/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Libraries imported successfully


In [3]:
# Environment, database, vector store, and display helpers are provided by langgraph_core
# (see langgraph_core.py). No local definitions needed for reusability.
print("✅ Using langgraph_core for environment, data sources, and display helpers")


✅ Using langgraph_core for environment, data sources, and display helpers


## 3️⃣ Setup Environment & Local Data Sources

Initialize:
- LangSmith tracing for observability
- Local SQLite database with sample portfolio data
- FAISS vector store with research documents

In [4]:
# Setup environment (loads API keys, enables LangSmith tracing)
config = setup_environment(
    langsmith_project="langgraph-bigdata-mcp-demo",
    enable_tracing=True
)

# Create local database with sample financial data
create_financial_database()

# Create vector store with research documents
create_vector_store()

print("\n✅ Local data sources ready")

✅ LangSmith tracing enabled → Project: langgraph-bigdata-mcp-demo
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...
✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

✅ Local data sources ready


## 4️⃣ Load Local Tools

Load tools that interact with local data sources:

In [5]:
# Get local database tools
local_db_tools = get_database_tools()
print(f"✅ Loaded {len(local_db_tools)} database tools:")
for t in local_db_tools:
    print(f"   - {t.name}: {t.description[:80]}...")

# Get local vector store tools
local_vector_tools = get_vectorstore_tools()
print(f"\n✅ Loaded {len(local_vector_tools)} vector store tools:")
for t in local_vector_tools:
    print(f"   - {t.name}: {t.description[:80]}...")

✅ Loaded 2 database tools:
   - internal_query_database: Execute SQL query against the internal financial transactions database.

Availab...
   - internal_portfolio_summary: Get a summary of a specific portfolio from internal database including holdings ...

✅ Loaded 1 vector store tools:
   - internal_search_research: Search internal research documents using semantic similarity.

This searches thr...


## 5️⃣ Connect to Bigdata MCP Server

**MCP Configuration:**
- **URL**: `https://mcp.bigdata.com/`
- **Transport**: HTTP (streamable)
- **Authentication**: `x-api-key` header

The MCP client automatically discovers all available tools from the server.

In [6]:
# API keys
BIGDATA_API_KEY = os.getenv("BIGDATA_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not BIGDATA_API_KEY:
    raise ValueError("BIGDATA_API_KEY not found. Set via: export BIGDATA_API_KEY='your-key'")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Set via: export OPENAI_API_KEY='your-key'")

print(f"✅ Bigdata API Key: {BIGDATA_API_KEY[:10]}...")
print(f"✅ OpenAI API Key: {OPENAI_API_KEY[:10]}...")

✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...


### Configure MCP Client

**Important**: Set `ANYIO_BACKEND='asyncio'` to ensure async backend detection in Jupyter:

In [7]:
# Set async backend for anyio (required in Jupyter)
os.environ['ANYIO_BACKEND'] = 'asyncio'

# Configure MCP client
mcp_client = MultiServerMCPClient(
    {
        "bigdata": {
            "url": "https://mcp.bigdata.com/",
            "transport": "http",  # Streamable HTTP transport
            "headers": {
                "x-api-key": BIGDATA_API_KEY
            }
        }
    }
)

print("✅ MCP client configured")

✅ MCP client configured


### Load MCP Tools

The MCP client automatically discovers all tools exposed by the Bigdata MCP server:

**Note**: In Jupyter notebooks, we use `await` directly instead of `asyncio.run()` because Jupyter already runs an event loop in the background.

In [8]:
# Load tools from Bigdata MCP server
# Note: In Jupyter notebooks, there's already a running event loop, so we use 'await' directly
print("Connecting to Bigdata MCP...")
bigdata_mcp_tools = await mcp_client.get_tools()
print(f"✅ Loaded {len(bigdata_mcp_tools)} tools from Bigdata MCP:")
for tool in bigdata_mcp_tools:
    print(f"   - {tool.name}: {tool.description[:80]}...")

Connecting to Bigdata MCP...
✅ Loaded 5 tools from Bigdata MCP:
   - bigdata_country_tearsheet: Returns a comprehensive country economic tearsheet with economic calendar data.
...
   - bigdata_events_calendar: Returns a professionally formatted markdown calendar of corporate events includi...
   - find_companies: REQUIRED FIRST STEP: Run this tool whenever the user mentions a company for the ...
   - bigdata_search: Search engine for financial documents, earnings call transcripts, news articles,...
   - bigdata_company_tearsheet: Returns a comprehensive company tearsheet with financial data, market intelligen...


## 6️⃣ Combine All Tools

Merge tools from all sources:

In [9]:
# Combine all tools
all_tools = local_db_tools + local_vector_tools + bigdata_mcp_tools

print(f"\n✅ Total tools available: {len(all_tools)}")
print(f"   - Local DB tools: {len(local_db_tools)}")
print(f"   - Local vector store tools: {len(local_vector_tools)}")
print(f"   - Bigdata MCP tools: {len(bigdata_mcp_tools)}")


✅ Total tools available: 8
   - Local DB tools: 2
   - Local vector store tools: 1
   - Bigdata MCP tools: 5


## 7️⃣ Create LangChain Agent

**LangChain ReAct Agent:**
- **Reasoning**: Plans which tools to use based on user query
- **Acting**: Executes tool calls and processes results
- **Iteration**: Continues until query is fully answered

Uses `langchain.agents.create_agent` (the current, non-deprecated API) which creates an agent with:
- Agent executor (LLM with tool calling capabilities)
- Tool registry (all available tools)
- System prompt (guides agent behavior)
- Streaming support (for real-time responses)

In [13]:
# Define system prompt for the agent
SYSTEM_PROMPT = """You are an intelligent financial research assistant with access to multiple data sources:

**External Data (Bigdata.com MCP):**
- Tools dynamically loaded from Bigdata MCP server
- Tools include news, prices, tear sheet, search, company lookup, and other capital markets capabilities

**Internal Data (Company Systems):**
- `internal_query_database` - Execute SQL queries on portfolio/transaction database
- `internal_portfolio_summary` - Get portfolio holdings and performance summary
- `internal_search_research` - Search internal investment research documents

Guidelines:
- Use appropriate tools based on the query
- For portfolio questions, use internal database tools
- For market intelligence, use Bigdata MCP tools
- Combine multiple sources for comprehensive analysis

**Citation format:** Use inline citations with the **source name as the link text** (not the raw URL). Format as markdown: [Source Name](url) or [1](url), [2](url) so the reader sees a clickable source name. Do not paste full URLs in the body.

**Do not add a separate "Sources" or "References" or "External sources" block at the end** when you have already used inline citations in the text. Inline citations are sufficient.
**Do not offer suggestions for follow up questions**

Available portfolios: PF001 (US Large Cap Growth), PF002 (AI & Semiconductor Focus), PF003 (Diversified Tech Leaders)
"""

model = "gpt-5"
# Initialize LLM
llm = ChatOpenAI(
    model=model,
    temperature=0,
    api_key=OPENAI_API_KEY
)

# Create agent using langchain.agents.create_agent (non-deprecated)
agent = langchain_create_agent(llm, all_tools, system_prompt=SYSTEM_PROMPT)

print(f"✅ Agent created with {len(all_tools)} tools")
print(f"   Model: {model}")
print(f"   System prompt configured")

✅ Agent created with 8 tools
   Model: gpt-5
   System prompt configured


## 8️⃣ Run Example Queries

Let's test the agent with queries that utilize different data sources:

### Example 1 : Multinode

In [11]:
query = """
Analyze the AI & Semiconductor Focus portfolio (PF002):
1. What are our current holdings and their performance?
2. What risks does our internal research identify?
3. For each holding, get us the pricing information from tearsheet
4. For each holding, get us negative news
"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

Here’s the PF002 review, organized per your requests.

1) Current holdings and performance (internal)
- NVIDIA (NVDA)
  - Shares: 12,000
  - MV: $10,506,000 (65.7% weight)
  - Avg cost: $450.00 | Current: $875.50
  - Unrealized P&L: +$5,106,000 (+94.6%)
- Broadcom (AVGO)
  - Shares: 1,500
  - MV: $2,137,500 (13.4% weight)
  - Avg cost: $850.00 | Current: $1,425.00
  - Unrealized P&L: +$862,500 (+67.6%)
- Palantir (PLTR)
  - Shares: 25,000
  - MV: $1,631,250 (10.2% weight)
  - Avg cost: $18.50 | Current: $65.25
  - Unrealized P&L: +$1,168,750 (+252.7%)
- AMD (AMD)
  - Shares: 8,000
  - MV: $1,162,000 (7.3% weight)
  - Avg cost: $95.00 | Current: $145.25
  - Unrealized P&L: +$402,000 (+52.9%)
- Taiwan Semi (TSM)
  - Shares: 3,000
  - MV: $557,250 (3.5% weight)
  - Avg cost: $110.00 | Current: $185.75
  - Unrealized P&L: +$227,250 (+68.9%)

Portfolio totals
- AUM (MV of holdings): $15,994,000
- Unrealized P&L: +$7,766,500
- Implied cost basis: $8,227,500
- Aggregate return (unrealized): +94.4%

2) Key risks flagged by internal research (synthesis)
Portfolio-level
- Valuation compression risk across AI leaders after a major run-up; earnings or AI-monetization disappointments could de-rate multiples (internal Technology Sector Risk Assessment, Jan-2025).
- US–China export controls and policy volatility: risk to data center AI hardware and supply chains; regulatory whiplash can disrupt shipments and revenue timing (esp. NVDA; internal NVDA thesis update).
- AI demand cyclicality/ROI uncertainty: enterprise AI spend may ebb/flow with macro and realized returns (internal strategy memos).
- Supply chain/supplier constraints: advanced memory and HBM supply tightness can cap upside for AI accelerators (NVDA/AMD).
- Customer concentration: hyperscaler budgets drive cycle amplitude (NVDA, AMD); large-customer dynamics in semis/software can amplify volatility.

Name-specific
- NVIDIA (NVDA): China exposure to export policy; potential share shift to AMD/custom silicon; supply constraints; regulatory oversight of dominant ecosystem (internal NVDA thesis).
- Broadcom (AVGO): Integration and execution risk post-VMware; regulatory/government actions affecting software footprint; AI mix potentially pressuring gross margins near term (internal sector views).
- Palantir (PLTR): Reputational/regulatory/privacy scrutiny tied to government contracts; contract timing lumpiness; valuation risk vs. growth trajectory (internal notes).
- AMD (AMD): CUDA/ROCm ecosystem gap vs. NVIDIA; MI300 ramp and supply tightness; valuation vs. execution risk (internal AMD thesis).
- TSMC (TSM): Taiwan geopolitical/seismic risk; overseas fab execution (US/Japan/Europe) with potentially lower returns; export-control uncertainty (internal macro/semis notes).

3) Pricing snapshot from company tearsheets (as of the latest tearsheet refresh; real-time quotes may differ by venue/currency)
- NVIDIA (NVDA) — USD
  - Last: $192.51 | Day chg: +$0.99 (+0.52%) | Mkt cap: $4.69T
  - 52w: $86.62–$212.19
- Broadcom (AVGO) — USD
  - Last: $330.73 | Day chg: -$2.51 (-0.75%) | Mkt cap: $1.57T
  - 52w: $138.10–$414.61
- Palantir (PLTR) — USD
  - Last: $151.86 | Day chg: -$5.49 (-3.49%) | Mkt cap: $346.9B
  - 52w: $66.12–$207.52
- AMD (AMD) — USD
  - Last: $252.18 | Day chg: -$0.56 (-0.22%) | Mkt cap: $410.6B
  - 52w: $76.48–$267.08
- Taiwan Semiconductor (TSM) — TWD (primary listing)
  - Last: NT$1,790 | Day chg: -NT$10 (-0.55%) | Mkt cap: NT$46,543B
  - 52w: NT$780–NT$1,830
Note: TSM quotes above are in TWD; ADR pricing in USD will differ. Some quotes may reflect post-split adjustments versus internal lot pricing.

4) Recent negative public news per holding (last ~30 days)
- NVIDIA (NVDA)
  - Reports that Chinese customs blocked H200 AI chips; suppliers paused certain component production pending clarity [MT Newswires](https://app.bigdata.com/files#?document=8BC27E648ABD8DDE329168138E7312D3), [Yahoo! News](https://sg.yahoo.com/finance/news/nvidia-stock-slides-china-blocks-113040268.html)
  - Tight HBM/memory supply could constrain export licenses to China under new rules, risking near-term sales [MT Newswires](https://app.bigdata.com/files#?document=63E91F23E9438FA2D6B94387E0A3937D), [MT Newswires](https://app.bigdata.com/files#?document=6B207F0D98917989B2ED9DCF83739BDC)
- Broadcom (AVGO)
  - China reportedly told firms to stop using certain foreign cybersecurity software, with VMware cited among affected products; AVGO shares fell on the headlines [Yahoo! Finance](https://finance.yahoo.com/news/broadcom-shares-plunge-report-china-121556714.html), [MT Newswires](https://app.bigdata.com/files#?document=51FE0EEFC59E8003CE085B5EAC58D608), [Benzinga](https://www.benzinga.com/node/49918896?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)
  - Valuation and margin concerns; commentary on softer AI gross margin mix and guidance sensitivity [Nasdaq](https://www.nasdaq.com/articles/1593-p-s-broadcom-overvalued-buy-sell-or-hold-stock)
- Palantir (PLTR)
  - Heightened scrutiny over ICE-related work and allegations around data usage (Medicaid data controversy); reputational and policy backlash risk [MSN](https://www.msn.com/en-gb/news/world/palantir-faces-fire-over-ice-use-of-medicaid-data/ar-AA1VflSb), [AOL.com](https://www.aol.com/news/palantir-courts-major-federal-contracts-170000722.html)
  - Analyst caution on potential deceleration in contract value metrics; downside price risk flagged [Yahoo! Finance](https://finance.yahoo.com/news/analyst-warns-palantir-stock-could-204308628.html)
- AMD (AMD)
  - No material negative public headlines of note identified across our monitored sources in the past 30 days.
- Taiwan Semi (TSM)
  - Earthquake-related production concerns resurfaced; ongoing geopolitical and export-control sensitivities around China exposure [Benzinga](https://www.benzinga.com/node/49681946?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), [Nasdaq](https://www.nasdaq.com/articles/tsm-hits-52-week-high-should-you-hold-stock-or-book-profits)
  - U.S. export license for China operations reported as temporary; continued policy risk and execution risks for overseas fabs (Arizona) noted in coverage [Yahoo! Finance](https://uk.finance.yahoo.com/news/tsmc-secures-one-us-export-023117235.html), [Miami Herald (MH)](https://www.miamiherald.com/news/business/article314365859.html#storylink=partnerdigest_the)

If you want, I can export this into a concise one-pager with links and a risk heat map.

### Example 2: Bigdata MCP Tool Query

Use external market intelligence:

In [14]:
# Example 2: Use Bigdata MCP tools for external data
query = "Find the latest news about NVIDIA's earnings and revenue growth using Bigdata tools."

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
#display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

In [15]:
# Display response
display_response(result)

# Display citations from Bigdata.com sources
display_citations(result)

Here’s the latest on NVIDIA’s earnings and revenue growth from recent Bigdata sources:

- Latest reported quarter (fiscal Q3 FY2026, reported Nov 19, 2025): Revenue was $57.0B (+62% YoY, +22% QoQ), with non-GAAP EPS of $1.30 (beat). Data Center led with $51.2B (+66% YoY, +25% QoQ). Management guided fiscal Q4 revenue to about $65B (±2%), implying ~14% sequential growth, and flagged very tight supply/“sold out” conditions for cloud GPUs. [Morningstar](https://global.morningstar.com/en-ca/stocks/nvidia-earnings-no-signs-near-term-ai-bubble-raising-fair-value), [Business Insider](https://markets.businessinsider.com/news/stocks/nvidia-ceo-says-ai-boom-is-real-not-a-bubble-after-strong-q3-quarter-1035580672), [MT Newswires](https://app.bigdata.com/files#?document=0187F910E4544D6964A28AD7342A3C95)

- Segment detail: Networking revenue accelerated sharply, up 162% YoY to $8.2B in the latest quarter, while Gaming rose ~30% and Automotive ~32%, underscoring that growth remains broad-based beyond core compute. [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-eyes-openai-investment-buy-182748007.html)

- Mix and margins: Data Center accounted for roughly 90% of total sales in the latest quarter. Gross margin was about 73% in Q3, with guidance later indicating mid-70s for Q4. [Nasdaq](https://www.nasdaq.com/articles/nvidias-q3-data-center-sales-soar-56-can-nvda-sustain-momentum), [Business Insider](https://markets.businessinsider.com/news/stocks/nvidia-ceo-says-ai-boom-is-real-not-a-bubble-after-strong-q3-quarter-1035580672)

- Demand visibility: Management commentary points to very strong multi-quarter demand; the CFO recently said AI demand has grown beyond NVIDIA’s prior ~$500B outlook, and the Q3 call noted visibility to “half a trillion” in Blackwell/Rubin-related revenue through 2026. [Business Insider](https://markets.businessinsider.com/news/stocks/nvidia-nvda-finance-chief-says-ai-demand-exceeds-500-billion-forecast-1035688759), [AOL.com](https://www.aol.com/finance/nvidia-nvda-q3-2026-earnings-231840117.html)

- What’s next: NVIDIA’s next earnings is slated for Feb 25, 2026. Street expectations center around ~$65.5B in revenue for fiscal Q4, implying ~66% YoY growth, consistent with the company’s guidance band. [Yahoo! Finance](https://finance.yahoo.com/news/nvidia-nvda-falls-more-steeply-225502967.html), [Nasdaq](https://www.nasdaq.com/articles/nvidia-nvda-falls-more-steeply-broader-market-what-investors-need-know)

If you’d like, I can also pull the real-time earnings day headlines and consensus changes as we get closer to Feb 25.

### Example 3: Multi-Source Query

Combine local and external data:

In [16]:
# Example 3: Multi-source comprehensive analysis
query = """For our NVIDIA holdings:
1. Check our internal database to see which portfolios hold NVDA and how much
2. Search our internal research for our investment thesis
3. Use Bigdata tools to find recent news about NVIDIA
4. Provide a comprehensive summary combining all sources"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

Here’s a consolidated view of our NVIDIA exposure, internal thesis, and the latest market intelligence.

1) Internal portfolios holding NVDA (per internal holdings database)
- PF002 – AI & Semiconductor Focus
  - Shares: 12,000
  - Avg cost: $450.00
  - Internal current price: $875.50
  - Market value: $10,506,000
  - Unrealized P&L: +$5,106,000
  - Position weight (vs. PF002 AUM $15.0M): ~70.0%
- PF003 – Diversified Tech Leaders
  - Shares: 8,000
  - Avg cost: $520.00
  - Internal current price: $875.50
  - Market value: $7,004,000
  - Unrealized P&L: +$2,844,000
  - Position weight (vs. PF003 AUM $50.0M): ~14.0%
- Portfolios without NVDA: PF001 holds no NVDA based on the current holdings query
- Firmwide (across PF002 and PF003): 20,000 shares; blended avg cost ~$478; total market value ~$17.51M; total unrealized P&L +$7.95M

2) Internal research highlights on NVIDIA (NVDA)
- Investment Thesis Update (Dec 15, 2024)
  - Core drivers: explosive Data Center growth (H100/H200 demand), roadmap to Blackwell (B100/B200) with material perf uplift, and durable CUDA software moat (switching costs, 4M+ developers)
  - AI inference TAM opportunity: large and growing as enterprise deployments scale
  - Key risks: China export restrictions, AMD competition, and supply constraints
  - Stance: Strong Buy; PT $950 (based on 25x FY26E EPS)
- Strategy memo (Jan 5, 2025)
  - Allocation: recommended +3% increase to NVDA on continued AI training demand/supply tightness; themes to monitor include enterprise inference scaling and cloud spend reacceleration
- Tech sector risk assessment (Jan 10, 2025)
  - Noted elevated valuation risk for “Mag 7” and specific NVDA risk from China exposure (20–25% revenue at risk from export controls); warned on potential AI capex/ROI mismatch if adoption lags

3) Recent NVIDIA news (last 7 days)
- Strategic investment and partnership expansion with CoreWeave
  - NVIDIA invested $2B in CoreWeave and expanded plans to help build >5GW of AI “factories” by 2030, strengthening a key channel/partner for AI infrastructure demand [MT Newswires](https://app.bigdata.com/files#?document=7DB1D89A240B51EB17C277E5F696ED88); additional coverage: [The Fly](https://app.bigdata.com/files#?document=8A18A14BF303B27A908A3D7C53C9847D)
- Autonomous/robotaxi ecosystem push
  - Mercedes-Benz said it will work with NVIDIA (and Uber) to develop a robotaxi network built around S-Class vehicles, highlighting automotive AI/computing adjacencies [MT Newswires](https://app.bigdata.com/files#?document=FAC2144CEF43B58817D782BD1CB7F258)
- China AI chips
  - Reuters-sourced report noted Alibaba, ByteDance, Tencent received approval to purchase >400,000 NVIDIA H200 chips, a potential tailwind for demand visibility in China [MT Newswires](https://app.bigdata.com/files#?document=47B7737C06B514353C732D96B703)
- AI and climate software
  - NVIDIA launched “Earth-2” open models, an open-sourced AI stack focused on weather/climate forecasting—supporting broader software ecosystem adoption and use cases [MT Newswires](https://app.bigdata.com/files#?document=951EF7D40D75940B10D22103004D1559)
- Ecosystem investing and partnerships
  - NVIDIA’s VC arm joined a $200M round for Synthesia (AI video platform), reinforcing NVIDIA’s ecosystem reach in AI applications [MT Newswires](https://app.bigdata.com/files#?document=91ED32DCD9D299774D36A4E17929FE61)
- Competitive dynamics and hyperscaler strategies
  - Commentary around Microsoft’s Maia 200 custom AI chips suggested a gradual reduction in reliance on NVIDIA at the margin, with knock-on implications for networking/infra players [The Fly](https://app.bigdata.com/files#?document=269B21373EA75E7201B3E2D220FB3C78)
- Sentiment/flows and expectations
  - Retail investors have been heavy net buyers of NVDA since mid-2025, indicating strong retail participation and sentiment [Benzinga](https://www.benzinga.com/node/50117320?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)
  - Mixed pre-earnings debate and targets ahead of NVIDIA’s scheduled Feb 25 earnings date [Yahoo! Finance](https://finance.yahoo.com/news/200-150-nvidia-february-25-110041394.html)

4) Synthesis and implications
- Portfolio exposure: NVDA is a concentrated driver in PF002 (~70% weight) and a significant position in PF003 (~14%). Mark-to-internal-price unrealized gains are substantial (+$7.95M across these two portfolios).
- Thesis alignment: Recent CoreWeave investment/partnership expansion extends the supply/demand flywheel for AI infrastructure—consistent with our thesis on data center acceleration and NVIDIA’s platform advantage (hardware + software). Earth-2 adds to software/research credibility and vertical AI adoption, while ecosystem investments (e.g., Synthesia) reinforce platform breadth.
- Risk balance: The report of China approvals for H200 chips is a near-term positive for demand visibility, though policy risk remains. Competitive signals from hyperscalers’ custom silicon (e.g., Microsoft Maia) point to a diversification trend; however, near-term constraints, software moats (CUDA), and breadth of NVIDIA’s roadmap continue to support share and pricing power. Elevated valuations keep sentiment-sensitive drawdown risk on the table, especially around major catalysts (e.g., the Feb 25 print).
- Monitoring priorities:
  - Supply ramp and lead times across H200 and Blackwell cycles
  - Hyperscaler capex mix (NVIDIA vs. custom/alternative accelerators) and inference spend trajectory
  - China export policy and realized shipments
  - Software monetization and platform stickiness (CUDA, enterprise stacks)
  - Concentration risk in PF002 given the outsized position weight

Notes
- Internal position data and valuations reflect our internal database snapshot (current_price field: $875.50). External price references in media may differ due to timing or share-split adjustments.

### Example 4: Company Briefing 

In [17]:
query = """
Summarize recent developments for CoreWeave (last 30 days).
**Steps:**
1. Call find_companies and get the company id
2. Call bigdata_tearsheet and get business context
3. Use bigdata_search and find news in the last 30 days
4. Categorize findings
**Categories:**
- Financial results
- Product/tech launches
- M&A and partnerships
- Regulatory/legal updates
- Management changes
- Other material events
For each: Date, facts, investment implications (bullish/bearish/neutral).
Please add inline source attribution.
"""

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)



In [18]:
# Display response
display_response(result)

# Display citations from Bigdata.com sources
#display_citations(result)

CoreWeave — last 30 days developments (through Jan 30, 2026)

Brief business context
- CoreWeave is an AI-focused cloud platform (“hyperscaler”) that provides GPU-accelerated compute and managed software for GenAI workloads, operating purpose-built data centers and partnering closely with NVIDIA. In the last month it disclosed an expanded collaboration with NVIDIA tied to large-scale “AI factory” buildouts and financing via a private placement [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm). The business model rents NVIDIA GPUs under multi-year contracts to AI customers such as OpenAI, Meta and Microsoft [Benzinga](https://www.benzinga.com/node/49977155?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).

Categorized developments

Financial results
- Date: No quarterly results in the last 30 days
  - Facts: No new earnings releases during the period. (Next earnings is scheduled in early March, outside the 30-day window.)
  - Investment implications: Neutral — no fresh P&L/cash flow data this month; focus shifted to financing/partnership and litigation headlines.

Product/tech launches
- Date: Jan 26, 2026
  - Facts: CoreWeave outlined plans to adopt NVIDIA CPU and storage platforms and deploy multiple NVIDIA infrastructure generations across its platform; it also flagged potential inclusion of CoreWeave software in NVIDIA reference architectures for cloud partners and enterprises, as part of the expanded collaboration [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm).
  - Investment implications: Bullish — deepens tech roadmap and potential distribution via NVIDIA-aligned reference architectures, but execution and capex ramp remain key risks.

M&A and partnerships
- Date: Jan 26, 2026
  - Facts: NVIDIA invested $2.0B in CoreWeave via a private placement at $87.20/share and the companies announced a broadened collaboration to accelerate buildout of over 5 GW of “AI factories” by 2030, with NVIDIA support to help secure land, power and shells [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/1769628/000176962826000044/crwv-20260123.htm). Following this, sell-side commentary highlighted scope for 2026 revenue estimates to move higher if capacity is delivered as planned [MT Newswires](https://app.bigdata.com/files#?document=188A5FE5C910F1A3858209730BD31EAC); [Benzinga](https://www.benzinga.com/node/50154287?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
  - Investment implications: Bullish — strategic and financial validation from NVIDIA plus potential supply chain advantages; positive signaling on scale-up, with execution and infrastructure timing still important.

Regulatory/legal updates
- Dates: Jan 13–29, 2026
  - Facts: Multiple securities class-action filings and shareholder law firm investigations were announced, largely tied to alleged disclosures around data center development delays in 2025 and the terminated Core Scientific merger, including Pomerantz filing a class action on Jan 29 [Benzinga](https://www.benzinga.com/node/50231987?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), Hagens Berman notice on Jan 21 [Benzinga](https://www.benzinga.com/node/50055833?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), and several investor alert deadlines from other firms between Jan 13–24 [Benzinga](https://www.benzinga.com/node/50116255?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack); [Associated Press](https://apnews.com/press-release/business-wire/coreweave-inc-crwv-shareholders-who-lost-money-contact-law-offices-of-howard-g-smith-about-securities-fraud-investigation-6eb1b23e596c4acdb5b81a1bfd733126).
  - Investment implications: Bearish — adds litigation and headline risk; potential for legal costs/management distraction, though outcomes and financial impact are uncertain and could take time to resolve.

Management changes
- Date: N/A (none observed)
  - Facts: No notable executive or board changes surfaced in the last 30 days based on available reports.
  - Investment implications: Neutral.

Other material events
- Date: Jan 27, 2026
  - Facts: Deutsche Bank upgraded the stock, citing a solid medium-term outlook heading into Q4 results and potential upside to 2026 consensus if capacity is delivered to contracted customers [MT Newswires](https://app.bigdata.com/files#?document=188A5FE5C910F1A3858209730BD31EAC).
  - Investment implications: Bullish — supportive sell-side stance contingent on execution of capacity rollouts.

- Date: Jan 27, 2026
  - Facts: Analysts increased forecasts following the NVIDIA collaboration and investment update [Benzinga](https://www.benzinga.com/node/50154287?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
  - Investment implications: Bullish — estimate revisions reflect improved confidence in scaling plans.

- Date: Jan 16, 2026
  - Facts: Shares rallied as investors interpreted TSMC’s guidance as supportive for AI chip supply and demand; article reiterated CoreWeave’s model of buying NVIDIA GPUs and renting capacity to customers like OpenAI/Meta/Microsoft [Benzinga](https://www.benzinga.com/node/49977155?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack).
  - Investment implications: Neutral to Bullish — better perceived supply/demand backdrop can support deliveries and backlog conversion, but not a company-specific operational update.

- Date: Jan 21, 2026
  - Facts: Coverage noted CoreWeave’s stance that prior data center delays do not reduce contract value; customers accepted adjusted delivery schedules, preserving backlog integrity [Nasdaq](https://www.nasdaq.com/articles/can-coreweave-convert-its-55b-backlog-profitable-growth).
  - Investment implications: Neutral to Bullish — suggests demand durability, though timing of capacity go-lives remains a key driver for recognized revenue and cash flow.

Notes on the last 30 days
- The period was defined by a major strategic-financing announcement with NVIDIA and a flurry of litigation notices related to 2025 disclosures and delays. Sell-side tone improved post-collaboration, but the investment case remains sensitive to buildout timing, capex intensity, and legal overhang.

### Example 5: Local Database Query

Query internal portfolio holdings:

In [19]:
# Example 5: Query internal database for top holdings
query = "What are our top 5 holdings by market value across all portfolios?"

# Display query
display_query(query)

# Run agent (async invocation in Jupyter)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)


Top 5 holdings by aggregated market value (across all portfolios):

1) NVIDIA (NVDA) — $17,510,000 — 20,000 shares
2) Microsoft (MSFT) — $9,556,500 — 23,000 shares
3) Apple (AAPL) — $7,410,000 — 40,000 shares
4) Salesforce (CRM) — $3,255,000 — 10,000 shares
5) Meta Platforms (META) — $2,632,500 — 4,500 shares

Notes:
- Values reflect the market_value field aggregated across PF001, PF002, and PF003 as recorded in the system.

### Example 6: Local Vector Store Query

Search internal research documents:

In [20]:
# Example 6: Search internal vector store
query = "What does our internal research say about NVIDIA's competitive moat?"

# Display query
display_query(query)

# Run agent (async invocation)
result = await agent.ainvoke({"messages": [("user", query)]})

# Display tools used
display_tools_used(result)

# Display response
display_response(result)


Here’s the consolidated view from our internal research on NVIDIA’s competitive moat:

What constitutes the moat
- Software ecosystem and switching costs: CUDA is the linchpin, with 4M+ developers and a mature stack (framework integrations, libraries, inference runtimes) that creates meaningful developer lock-in and workload portability advantages. This remains the primary barrier to entry versus rivals whose ecosystems are less mature (Investment Thesis Update – Dec 15, 2024; AMD Assessment – Dec 11, 2024).
- Performance leadership and product cadence: NVIDIA’s rapid cadence from Hopper (H100/H200) to Blackwell (B100/B200, targeted ~2.5x performance uplift) underpins a lead in AI training/inference efficiency and time-to-deploy, reinforcing the ecosystem flywheel (Investment Thesis Update – Dec 15, 2024).
- Scale and integration advantages: Explosive data center growth and a full-stack approach (silicon + systems + software) support network effects with hyperscalers and enterprises, reinforcing mindshare and standards leadership (Investment Thesis Update – Dec 15, 2024; Portfolio Strategy – Jan 5, 2025).
- Distribution and partnerships: Deep hyperscaler partnerships and broad cloud availability accelerate developer adoption and enterprise deployment, compounding the software moat (Portfolio Strategy – Jan 5, 2025).
- Supply chain positioning: Persistent demand exceeding supply highlights NVIDIA’s advantaged access and coordination across leading-edge manufacturing/packaging and memory, which is itself a temporary barrier to rivals catching up at scale (Investment Thesis Update – Dec 15, 2024).

How we see the risks to the moat
- Competitive catch-up: AMD’s MI300X is competitive on memory capacity and inference in select LLM workloads, but ROCm still trails CUDA in breadth and maturity; mindshare remains a NVIDIA advantage (AMD Assessment – Dec 11, 2024).
- Custom silicon: Hyperscaler-specific accelerators (e.g., TPUs/Trainium) can erode share in targeted workloads, particularly inference, though they don’t yet match CUDA’s broad developer base (cross-doc synthesis).
- Policy and geopolitics: Export controls to China represent a material revenue risk (internally estimated at 20–25%) and could disrupt demand mix and allocation (Technology Sector Risk Assessment – Jan 10, 2025).
- Macro/ROI risk: AI infrastructure spend could outpace near-term monetization, creating the risk of digestion periods that test pricing power and shipment visibility (Technology Sector Risk Assessment – Jan 10, 2025).

Durability and outlook
- Near-to-medium term (12–24 months): We view the moat as durable through the Blackwell cycle, anchored by CUDA switching costs, developer mindshare, and product cadence. Key watch items are ROCm maturity, the pace of custom accelerator adoption at major clouds, and any sustained easing of supply tightness (Investment Thesis Update – Dec 15, 2024; Portfolio Strategy – Jan 5, 2025).

Investment stance in our materials
- We maintain a constructive view: Strong Buy with a $950 PT as of Dec 15, 2024, citing the software moat and product leadership; we also recommended a +3% portfolio overweight in January 2025 on training demand exceeding supply (Investment Thesis Update – Dec 15, 2024; Portfolio Strategy – Jan 5, 2025).

## 9️⃣ Understanding the Agent Flow

The agent follows this process:

1. **Parse Query** → Understand what information is needed
2. **Plan** → Decide which tools to use
3. **Execute** → Call selected tools in sequence or parallel
4. **Synthesize** → Combine results into coherent answer
5. **Iterate** → If more information needed, repeat steps 2-4

**Tool Selection Logic:**
- Portfolio/holdings questions → `internal_query_database` or `internal_portfolio_summary`
- Internal research → `internal_search_research`
- External market data → Bigdata MCP tools (automatically discovered)

**Trace Visibility:**
- All tool calls are logged to LangSmith for debugging
- View traces at: https://smith.langchain.com

## 🔟 Key Benefits of This Architecture

### 1. **Automatic Tool Discovery**
- MCP server exposes tools dynamically
- No code changes needed when Bigdata.com adds new tools
- Agent automatically learns about new capabilities

### 2. **Unified Interface**
- Single agent interface for all data sources
- Consistent tool calling pattern
- Easy to add more MCP servers or local tools

### 3. **Stateful Reasoning**
- LangGraph maintains conversation state
- Agent can reference previous tool results
- Multi-turn reasoning supported

### 4. **Observability**
- LangSmith tracing shows full execution flow
- Easy to debug tool selection and results
- Performance monitoring built-in

## 🎯 Next Steps

**Extend this architecture:**

1. **Add More MCP Servers**
   ```python
   mcp_client = MultiServerMCPClient({
       "bigdata": {...},
       "other_mcp_server": {...}
   })
   ```

2. **Custom Local Tools**
   - Create @tool decorated functions
   - Add to tool list before agent creation

3. **Advanced Graph Patterns**
   - Use `StateGraph` for custom control flow
   - Add conditional edges for routing logic
   - Implement human-in-the-loop

4. **Persistent Memory**
   - Add checkpointer for conversation history
   - Use `MemorySaver` or Redis for state persistence

**References:**
- LangGraph: https://langchain-ai.github.io/langgraph/
- MCP Adapters: https://reference.langchain.com/python/langchain_mcp_adapters/
- Bigdata MCP: https://docs.bigdata.com/mcp-reference/

---

## 📚 Additional Resources

- **Bigdata.com API Documentation**: https://docs.bigdata.com
- **LangGraph Documentation**: https://langchain-ai.github.io/langgraph/
- **MCP Protocol Spec**: https://modelcontextprotocol.io
- **LangSmith Tracing**: https://smith.langchain.com

**Questions?** Contact: support@bigdata.com